# nanoWM M2 pre-ticket profile
Profiles only the same-depth, fixed-head-count muP proxy/target pair. This is a capped microbenchmark, not an M2 training run and does not spend the training ledger.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
inputs = Path('/kaggle/input')
print('top-level /kaggle/input entries:', list(inputs.iterdir()))
roots = list((inputs / 'nanowm-code').rglob('pyproject.toml'))
if not roots:
    roots = list(inputs.rglob('pyproject.toml'))
if len(roots) != 1:
    raise RuntimeError(f'Expected exactly one project root, found {roots}')
mounted = roots[0].parent
root = Path('/kaggle/working/project')
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob('*.zip'):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
subprocess.run([sys.executable, 'scripts/preflight_gpu.py'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q'], check=True)

In [ ]:
out = '/kaggle/working/m2_profile.json'
subprocess.run([
    sys.executable, 'scripts/profile_dit.py',
    '--presets', 'm2_proxy_5m', '15m',
    '--tokens', '16', '--batch-size', '8',
    '--steps', '20', '--warmup', '3', '--time-budget', '20',
    '--compile-only', '--num-heads', '8', '--mup-base-dim', '192',
    '--out', out,
], check=True)

In [ ]:
import json
data = json.load(open('/kaggle/working/m2_profile.json'))
assert len(data['results']) == 2, data
for result in data['results']:
    assert not result.get('error') and not result.get('oom'), result
    assert result['compiled'], result
    assert result['nonfinite_losses'] == 0 and result['gradscaler_skips'] == 0, result
    print(result['preset'], result['params'], f"{result['step_time_median_s']*1000:.1f} ms", f"{result['peak_vram_gb']:.2f} GiB")
seconds = 5 * 3000 * sum(r['step_time_median_s'] for r in data['results'])
print(f'10-arm, 3000-step estimate: {seconds/60:.1f} wall min = {seconds/3600:.3f} GPU-h (single GPU), before setup margin')